# Gravity Inverse Modeling
Gravity Method Tutorial #5

<b> Dicky Ahmad Zaky </b>
<br> Geophysical Engineering Department
<br> Universitas Pertamina

#### 1. Introduction

In previous tutorials, we performed **Forward Modeling**: given subsurface model parameters $\mathbf{m} = [x_0, z, R, \Delta\rho]$, we calculated the predicted gravity response $\mathbf{d}_{\text{calc}} = f(\mathbf{m})$.

In exploration geophysics, the practical challenge is the **Inverse Problem**: given measured surface gravity anomaly data $\mathbf{d}_{\text{obs}}$, we wish to determine the subsurface model parameters $\mathbf{m}$ that produced them.

$$\mathbf{d}_{\text{obs}} \xrightarrow{\text{Inversion}} \mathbf{m} = \begin{bmatrix} x_0 \\ z \\ M_{\text{eff}} \end{bmatrix}$$

##### The Fundamental Challenges of Gravity Inversion:
1. **Non-Linearity:** The vertical gravity attraction over a buried body depends non-linearly on depth $z$ and position $x_0$:
   $$g(x) = \frac{G \cdot M_{\text{eff}} \cdot z}{\left((x - x_0)^2 + z^2\right)^{3/2}}$$
2. **Ambiguity / Non-Uniqueness (Gauss's Theorem):** Infinite combinations of density contrast $\Delta\rho$ and volume $V$ (or radius $R$) produce the exact same effective anomalous mass:
   $$M_{\text{eff}} = \Delta\rho \cdot V = \Delta\rho \cdot \frac{4}{3}\pi R^3$$
   Therefore, gravity alone can determine the *mass excess* $M_{\text{eff}}$, position $x_0$, and depth $z$. Constraining density or geometry independently requires geological priors or borehole/seismic information.

In this tutorial, we will formulate and execute a **Non-Linear Least Squares Inversion** using the Trust Region Reflective (Levenberg-Marquardt) algorithm from `scipy.optimize.curve_fit`.

#### 2. Forward Function Definition and Synthetic Data Generation

We model a buried spherical body with true parameters:
* True horizontal position: $x_0 = 2500\text{ m}$
* True depth to center: $z = 350\text{ m}$
* True radius: $R = 80\text{ m}$
* True density contrast: $\Delta\rho = 600\text{ kg/m}^3$

We then compute the true effective mass:
$$M_{\text{eff}} = \Delta\rho \cdot \frac{4}{3}\pi R^3 \approx 1.2868 \times 10^9\text{ kg}$$

We add 5% Gaussian noise to simulate real field data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# Universal Gravitational Constant
G = 6.67430e-11

# Analytical Forward Function for a Buried Sphere
def gravity_sphere_forward(x_stations, x0, z, M_eff):
    """
    Computes gravity anomaly (mGal) for a sphere with:
    - x0: horizontal center (m)
    - z: depth to center (m)
    - M_eff: anomalous mass (kg)
    """
    r_cubed = ((x_stations - x0)**2 + z**2)**1.5
    # g = (G * M_eff * z) / r^3  [m/s^2] -> * 1e5 [mGal]
    return (G * M_eff * z / r_cubed) * 1e5

# 1. Survey Profile
x_stations = np.arange(0, 5000 + 25, 25)  # 201 stations every 25 m

# 2. True Model Parameters
true_x0 = 2500.0          # m
true_z = 350.0            # m
true_R = 80.0             # m
true_drho = 600.0         # kg/m^3
true_V = (4.0 / 3.0) * np.pi * (true_R**3)
true_M_eff = true_drho * true_V  # kg

# 3. Compute True Forward Anomaly
g_clean = gravity_sphere_forward(x_stations, true_x0, true_z, true_M_eff)

# 4. Add 5% Noise
np.random.seed(2026)
noise_sigma = 0.05 * g_clean.max()
noise = np.random.normal(0, noise_sigma, size=len(x_stations))
g_observed = g_clean + noise

print(f"True Parameters:")
print(f"  Horizontal Center (x0): {true_x0:.1f} m")
print(f"  Depth (z)             : {true_z:.1f} m")
print(f"  Effective Mass (M_eff): {true_M_eff:.3e} kg")
print(f"  Peak Anomaly          : {g_clean.max():.3f} mGal")
print(f"  Noise std dev (sigma) : {noise_sigma:.4f} mGal")

#### 3. Defining Inversion Bounds and Initial Guess

Non-linear optimization iteratively searches parameter space starting from an **initial guess $\mathbf{m}_0$**.
A poor initial guess without bounds can cause divergence or convergence to non-physical solutions (e.g., negative depth).

We provide:
* **Initial Guess:** Deliberately offset from true values:
  * $x_0^{(0)} = 2200\text{ m}$ (off by 300 m)
  * $z^{(0)} = 600\text{ m}$ (off by 250 m)
  * $M_{\text{eff}}^{(0)} = 3.0 \times 10^9\text{ kg}$ (off by >200%)
* **Physical Bounds:**
  * $x_0 \in [500, 4500]\text{ m}$
  * $z \in [50, 1500]\text{ m}$
  * $M_{\text{eff}} \in [10^7, 10^{11}]\text{ kg}$

In [ ]:
# Initial Guess: [x0, z, M_eff]
initial_guess = [2200.0, 600.0, 3.0e9]

# Physical bounds: ([lower bounds], [upper bounds])
bounds = (
    [500.0, 50.0, 1.0e7],      # Lower bounds
    [4500.0, 1500.0, 1.0e11]   # Upper bounds
)

# Response of the Initial Guess (before inversion)
g_initial = gravity_sphere_forward(x_stations, *initial_guess)

print("Initial Guess Parameters:")
print(f"  x0_init   = {initial_guess[0]:.1f} m")
print(f"  z_init    = {initial_guess[1]:.1f} m")
print(f"  M_eff_init= {initial_guess[2]:.3e} kg")

#### 4. Executing the Non-Linear Least Squares Inversion

We run `scipy.optimize.curve_fit` which minimizes the sum of squared residuals:
$$\Phi(\mathbf{m}) = \sum_{i=1}^{M} \left( g_{\text{obs}}(x_i) - g_{\text{calc}}(x_i; \mathbf{m}) \right)^2$$

In [ ]:
# Perform Inversion
popt, pcov = curve_fit(
    gravity_sphere_forward,
    x_stations,
    g_observed,
    p0=initial_guess,
    bounds=bounds,
    sigma=np.full_like(g_observed, noise_sigma),
    absolute_sigma=True,
    method='trf'  # Trust Region Reflective algorithm
)

inv_x0, inv_z, inv_M_eff = popt

# Parameter Uncertainties (Standard Deviations from Covariance Matrix)
param_std = np.sqrt(np.diag(pcov))

# Inverted Model Forward Response
g_inverted = gravity_sphere_forward(x_stations, *popt)

# Compute Misfit Metrics
rmse_initial = np.sqrt(np.mean((g_observed - g_initial)**2))
rmse_inverted = np.sqrt(np.mean((g_observed - g_inverted)**2))

print("=" * 50)
print("INVERSION RESULTS AND COMPARISON:")
print("=" * 50)
print(f"Parameter        | True Value   | Inverted (+- std)          | Error (%)")
print("-" * 65)
print(f"x0 (m)           | {true_x0:10.1f}   | {inv_x0:8.2f} +- {param_std[0]:.2f}        | {abs(inv_x0-true_x0)/true_x0*100:6.2f}%")
print(f"z  (m)           | {true_z:10.1f}   | {inv_z:8.2f} +- {param_std[1]:.2f}        | {abs(inv_z-true_z)/true_z*100:6.2f}%")
print(f"M_eff (kg)       | {true_M_eff:10.3e}   | {inv_M_eff:8.3e} +- {param_std[2]:.2e}    | {abs(inv_M_eff-true_M_eff)/true_M_eff*100:6.2f}%")
print("-" * 65)
print(f"Initial RMS Misfit : {rmse_initial:.4f} mGal")
print(f"Inverted RMS Misfit: {rmse_inverted:.4f} mGal (Noise floor: {noise_sigma:.4f} mGal)")
print("=" * 50)

#### 5. Recovering Geometric Parameters and Addressing Non-Uniqueness

Suppose geological field samples indicate the host rock has a known regional density $\rho_0 = 2670\text{ kg/m}^3$ and the ore deposit has an average density $\rho_{\text{ore}} = 3270\text{ kg/m}^3$ (giving $\Delta\rho = +600\text{ kg/m}^3$).
With this geological prior, we can uniquely resolve the radius $R$ of the ore body from the inverted anomalous mass $M_{\text{eff}}$:

$$R_{\text{estimated}} = \left( \frac{3 \cdot M_{\text{eff}}}{4\pi \cdot \Delta\rho} \right)^{1/3}$$

In [ ]:
# Using density contrast prior to estimate body radius
assumed_drho = 600.0  # kg/m^3
inv_R = ((3.0 * inv_M_eff) / (4.0 * np.pi * assumed_drho))**(1.0 / 3.0)

print(f"Estimated Radius (with assumed drho = {assumed_drho} kg/m^3):")
print(f"  R_inverted = {inv_R:.2f} m  (True R = {true_R:.2f} m, Error = {abs(inv_R - true_R)/true_R*100:.2f}%)")

#### 6. Visualization: Inversion Fit and Subsurface Geometry

Now we plot the observed data against the initial model and inverted model, alongside a subsurface cross-section showing the true body vs the inverted body.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 8), sharex=True, gridspec_kw={'height_ratios': [2, 1.3]})

# Top Plot: Gravity Anomaly Fit
ax1.errorbar(x_stations[::4], g_observed[::4], yerr=noise_sigma, fmt='ko', markersize=3, alpha=0.6, label='Observed Data (with 5% noise)')
ax1.plot(x_stations, g_clean, 'k-', linewidth=1.5, alpha=0.5, label='True Synthetic Model')
ax1.plot(x_stations, g_initial, 'r:', linewidth=2, label=f'Initial Guess (RMSE={rmse_initial:.3f} mGal)')
ax1.plot(x_stations, g_inverted, 'b-', linewidth=2.5, label=f'Inverted Result (RMSE={rmse_inverted:.3f} mGal)')
ax1.set_ylabel('Gravity Anomaly (mGal)', fontsize=11)
ax1.set_title('Non-Linear Inversion Fit: Observed vs Initial vs Inverted Model', fontsize=12, fontweight='bold')
ax1.grid(True, linestyle='--', alpha=0.6)
ax1.legend(loc='upper right', frameon=True)

# Bottom Plot: Subsurface Geometry Comparison
theta = np.linspace(0, 2*np.pi, 200)

# Initial Model Body (R estimated assuming same drho)
init_R = ((3.0 * initial_guess[2]) / (4.0 * np.pi * assumed_drho))**(1.0 / 3.0)
ax2.plot(initial_guess[0] + init_R*np.cos(theta), initial_guess[1] + init_R*np.sin(theta), 'r:', linewidth=2, label='Initial Guess Geometry')

# True Body
ax2.fill(true_x0 + true_R*np.cos(theta), true_z + true_R*np.sin(theta), color='gray', alpha=0.5, label=f'True Sphere (z={true_z:.0f}m, R={true_R:.0f}m)')

# Inverted Body
ax2.plot(inv_x0 + inv_R*np.cos(theta), inv_z + inv_R*np.sin(theta), 'b-', linewidth=2.5, label=f'Inverted Sphere (z={inv_z:.0f}m, R={inv_R:.0f}m)')

ax2.set_xlabel('Profile Distance (m)', fontsize=11)
ax2.set_ylabel('Depth (m)', fontsize=11)
ax2.set_ylim(850, 0)  # Invert depth axis
ax2.set_xlim(0, 5000)
ax2.grid(True, linestyle='--', alpha=0.6)
ax2.legend(loc='lower right', frameon=True)

plt.tight_layout()
plt.show()

#### 7. Exercises for Students

1. **Robustness to Initial Guesses (Local Minima):**
   * Change the initial guess to $x_0^{(0)} = 1000\text{ m}$, $z^{(0)} = 1200\text{ m}$. Does the inversion still converge to the correct global minimum, or does it get stuck? Why is a good initial guess critical in non-linear geophysical inversion?

2. **Noise Thresholding:**
   * Increase the noise from 5% to 20% (`noise_sigma = 0.20 * g_clean.max()`).
   * Observe how the estimated parameter standard deviations (`param_std`) change. Which parameter is most sensitive to noise?

3. **Demonstrating Equivalence (Non-Uniqueness):**
   * Calculate the forward response for a sphere with $\Delta\rho = 300\text{ kg/m}^3$ and $R = 100.8\text{ m}$ placed at the same depth ($z=350\text{ m}$).
   * Compare its anomaly curve with our true model ($\Delta\rho = 600\text{ kg/m}^3, R=80\text{ m}$). 
   * What do you observe about the two curves? Discuss why gravity data must be integrated with other geoscientific constraints.

In [ ]:
# Workspace for student exercise code